# DeviceMesh 入门

本笔记根据 [PyTorch 官方教程：DeviceMesh 入门](https://docs.pytorch.ac.cn/tutorials/recipes/distributed_device_mesh.html) 整理。`DeviceMesh` 用一个多维逻辑网格描述集群中进程与设备的拓扑，并统一管理并行策略所需的通信组。

## 学习目标

完成后应能：

1. 用 `init_device_mesh()` 创建一维和多维设备网格。
2. 用 mesh 维度名获取对应的 `ProcessGroup`。
3. 用二维 mesh 配置 HSDP（FSDP 分片 + DDP 复制）。
4. 从三维父 mesh 切出子 mesh，以组合 HSDP 与张量并行。

## 1. 为什么需要 DeviceMesh？

大规模分布式训练通常会混合多种并行维度。例如二维并行中，一组 rank 用于参数分片，另一组 rank 用于参数副本同步。若手工实现，需要：

- 根据全局 `rank` 和 `world_size` 计算拓扑；
- 手工创建多个 `ProcessGroup`；
- 为每个 rank 找到它所属的通信组；
- 维护设备、rank 与并行维度之间的对应关系。

`DeviceMesh` 将这些拓扑信息集中成一个对象。并行 API 接受 mesh 后，可以按 mesh 维度自动使用相应通信组。

## 2. 启动前提

本教程的 CUDA 示例需要使用 `torchrun` 启动多个进程。以单机 8 卡为例：

```powershell
torchrun --standalone --nproc_per_node=8 your_script.py
```

每个进程对应一张 GPU。`torchrun` 会设置 `RANK`、`LOCAL_RANK`、`WORLD_SIZE` 等环境变量；`init_device_mesh()` 会基于默认分布式环境创建 mesh。

## 3. 创建一维 DeviceMesh

一维 mesh 最适合只有一个并行维度的场景，例如单纯的数据并行或单纯的张量并行。形状 `(8,)` 表示 8 个 rank 构成一条逻辑线：

```text
[rank 0, rank 1, rank 2, rank 3, rank 4, rank 5, rank 6, rank 7]
```

In [ ]:
from torch.distributed.device_mesh import init_device_mesh

# 所有 8 个 rank 构成一个一维 CUDA mesh。
mesh_1d = init_device_mesh("cuda", (8,))
print(mesh_1d)

## 4. 二维 mesh：以一套拓扑管理两类通信组

形状 `(2, 4)` 将 8 个 rank 组织为二维网格：

```text
                 shard 维度
             0       1       2       3
replicate  0   rank 0  rank 1  rank 2  rank 3
维度       1   rank 4  rank 5  rank 6  rank 7
```

- `replicate` 维度的组：`(0, 4)`、`(1, 5)`、`(2, 6)`、`(3, 7)`。
- `shard` 维度的组：`(0, 1, 2, 3)`、`(4, 5, 6, 7)`。

没有 DeviceMesh 时，需要手工调用 `dist.new_group()` 并根据 rank 选择正确组；有了 DeviceMesh，只需声明 mesh 形状和维度名称。

In [ ]:
from torch.distributed.device_mesh import init_device_mesh

mesh_2d = init_device_mesh(
    "cuda",
    (2, 4),
    mesh_dim_names=("replicate", "shard"),
)

# 取得当前 rank 在指定 mesh 维度上对应的通信组。
replicate_group = mesh_2d.get_group(mesh_dim="replicate")
shard_group = mesh_2d.get_group(mesh_dim="shard")

## 5. 使用 DeviceMesh 配置 HSDP

HSDP（Hybrid Sharding Data Parallel）是一种二维数据并行：

- 在 `dp_shard` 维度内执行 FSDP 参数分片；
- 在 `dp_replicate` 维度间执行 DDP 式副本同步。

将二维 mesh 传给 FSDP 后，FSDP 能从维度名称与拓扑中获得所需通信组，无需用户手工维护。

In [ ]:
import torch.nn as nn
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.fsdp import fully_shard as FSDP

class ToyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net1 = nn.Linear(10, 10)
        self.relu = nn.ReLU()
        self.net2 = nn.Linear(10, 5)

    def forward(self, x):
        return self.net2(self.relu(self.net1(x)))

# 2 x 4：一个维度负责副本同步，另一个维度负责参数分片。
hsdp_mesh = init_device_mesh(
    "cuda",
    (2, 4),
    mesh_dim_names=("dp_replicate", "dp_shard"),
)

model = FSDP(ToyModel(), device_mesh=hsdp_mesh)

## 6. 从父 mesh 切出子 mesh：组合并行

更复杂的训练可使用三维 mesh，例如 `(2, 2, 2)`：

- `replicate` + `shard` 两个维度组成 HSDP 子 mesh；
- `tp` 维度组成张量并行子 mesh。

切出的子 mesh 会复用父 mesh 已创建的 NCCL 通信器，因此不需要为每种组合并行方案重复创建通信组。

In [ ]:
from torch.distributed.device_mesh import init_device_mesh

mesh_3d = init_device_mesh(
    "cuda",
    (2, 2, 2),
    mesh_dim_names=("replicate", "shard", "tp"),
)

# 选择多个维度，得到 HSDP 使用的二维子 mesh。
hsdp_mesh = mesh_3d["replicate", "shard"]

# 只选择 tp 维度，得到张量并行使用的一维子 mesh。
tp_mesh = mesh_3d["tp"]

# 需要底层通信组时仍可访问。
replicate_group = hsdp_mesh["replicate"].get_group()
shard_group = hsdp_mesh["shard"].get_group()
tp_group = tp_mesh.get_group()

## 7. 要点回顾

- `DeviceMesh` 描述的是 **设备/进程拓扑**，不是张量本身的分片规则；DTensor 的 `placements` 才描述张量布局。
- mesh 的形状乘积必须等于参与进程数，例如 `(2, 4)` 需要 8 个 rank。
- 为 mesh 维度命名后，可以按名称切分子 mesh 或获取通信组，代码更不易出错。
- HSDP 和张量并行等策略可以共享一个多维父 mesh，实现组合并行。

下一步可结合同目录的 `DTensor.ipynb` 与 `dtensor_example.py`，继续学习如何用 DeviceMesh 为 DTensor 指定 `Shard`、`Replicate` 和 `Partial` 布局。